# 03 — Training: benchmark + LogReg / RF / XGBoost in SageMaker

**Team Update 4.1 steps 1–2.** Runs the simple benchmark, then trains the
three design-doc candidates as SageMaker Training jobs on the temporally-split
feature table. The champion selection happens in notebook 04.

| Step | Verify (📸) |
|---|---|
| Split materialization (Task 8) | 4 split CSVs in S3 |
| Benchmark on the full table | benchmark metrics printed |
| Training jobs | completed-job table + metrics comparison |

**Run order:** top to bottom in SageMaker Studio. Training uses
`ml.m5.2xlarge` (XGBoost) and `ml.m5.xlarge` (sklearn) per the design doc;
three seeded XGBoost runs over the design-doc grid.

## 1. Setup

In [ ]:
%pip install -q "sagemaker<3" xgboost

import json
import os
import tarfile
from pathlib import Path

import boto3
import pandas as pd
import sagemaker
from sagemaker import image_uris
from sagemaker.inputs import TrainingInput
from sagemaker.session import Session
from sagemaker.sklearn import SKLearn
from sagemaker.estimator import Estimator

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sm_client = boto_session.client("sagemaker", region_name=region)
sess = Session(boto_session=boto_session, sagemaker_client=sm_client)
try:
    role = sagemaker.get_execution_role()  # Studio / notebook instance
except Exception:
    # local run (personal account): dedicated project role if present,
    # otherwise the newest SageMaker execution role in the account
    iam = boto3.client("iam", region_name=region)
    by_name = {r["RoleName"]: r for r in iam.list_roles()["Roles"]}
    if "aai-540-g1-execution-role" in by_name:
        picked = "aai-540-g1-execution-role"
    else:
        cands = [n for n in by_name if n.startswith("AmazonSageMaker-ExecutionRole")]
        if not cands:
            raise RuntimeError("no SageMaker execution role found - run scripts/ensure_execution_role.py")
        picked = sorted(cands, key=lambda n: by_name[n]["CreateDate"])[-1]
    account = boto3.client("sts").get_caller_identity()["Account"]
    role = f"arn:aws:iam::{account}:role/{picked}"
print(f"role={role}")
bucket = sess.default_bucket()
prefix = "aai-540-g1"
s3 = boto3.client("s3", region_name=region)

print(f"role={role}\nbucket={bucket}")

## 2. Ensure the repo code is present in Studio

The notebook drives `src/train.py` / `src/benchmark.py` from the repo. Clone
it if this Studio file view doesn't already have it.

In [ ]:
import os
from pathlib import Path

# CWD depends on the runner: Studio file view starts at the repo root, while
# `jupyter nbconvert` starts in the notebook's directory. Resolve the repo
# root (the dir that contains src/) and work from there.
if (Path.cwd() / "src").is_dir():
    REPO = Path.cwd()
elif (Path.cwd().parent / "src").is_dir():
    REPO = Path.cwd().parent
else:  # fresh Studio file view: clone the repo next to the notebooks dir
    !git clone --quiet https://github.com/University-San-Diego-MAAI/AAI-540-Group-1-Final-Project.git ../repo
    REPO = Path.cwd().parent / "repo"
os.chdir(REPO)
print("repo root:", REPO, "| src/train.py:", (REPO / "src" / "train.py").is_file())

## 3. Task 8 — materialize the four split CSVs to S3

Feature-store-style dataset extraction: the training table carries the course
`split` column; we persist each slice as its own CSV so downstream jobs and
monitors can reference a stable per-split URI.

In [ ]:
s3.download_file(bucket, f"{prefix}/processed/training/training_table.csv", "training_table.csv")
df = pd.read_csv("training_table.csv")

for split_name in ["train", "validation", "test", "production"]:
    part = df[df["split"] == split_name]
    local = f"split_{split_name}.csv"
    part.to_csv(local, index=False)
    s3.upload_file(local, bucket, f"{prefix}/processed/splits/{local}")
    print(f"{split_name}: {len(part):,} rows -> s3://{bucket}/{prefix}/processed/splits/{local}")

## 4. Team Update 4.1 step 1 — benchmark on the full table

Simple heuristics only (majority class + history-visits tercile rule). The
real models must beat these on the test split.

In [ ]:
!python src/benchmark.py --table training_table.csv --output-dir artifacts/evaluation
s3.upload_file("artifacts/evaluation/benchmark_metrics.json", bucket,
               f"{prefix}/artifacts/evaluation/benchmark_metrics.json")

## 5. Training runs

`src/train.py` reads the full table from the `train` channel, derives the
splits internally, early-stops XGBoost on the validation slice, and writes
`<model>_metrics.json` into the model artifact. Hyperparameter keys use
dashes because SageMaker passes them to argparse verbatim.

In [ ]:
import time

TRAIN_CHANNEL = TrainingInput(
    f"s3://{bucket}/{prefix}/processed/training/training_table.csv",
    content_type="text/csv",
)
XGB_IMAGE = image_uris.retrieve("xgboost", region, version="1.7-1")

RUN_SUFFIX = str(int(time.time()))

XGB_GRID = {
    "xgb-e01-d6-w5":  {"eta": "0.1",  "max-depth": "6", "min-child-weight": "5"},
    "xgb-e005-d6-w5": {"eta": "0.05", "max-depth": "6", "min-child-weight": "5"},
    "xgb-e01-d8-w1":  {"eta": "0.1",  "max-depth": "8", "min-child-weight": "1"},
}


def run_training_job(name, estimator_cls, hp, instance, image=None):
    kwargs = dict(
        entry_point="train.py",  # relative to source_dir="src"
        source_dir="src",
        role=role,
        instance_count=1,
        instance_type=instance,
        output_path=f"s3://{bucket}/{prefix}/artifacts/training/{name}",
        sagemaker_session=sess,
        hyperparameters={"model": hp.pop("model"), **hp},
    )
    if estimator_cls is SKLearn:
        est = SKLearn(framework_version="0.23-1", **kwargs)
    else:
        est = Estimator(image_uri=image, **kwargs)
    job_name = f"{name}-{RUN_SUFFIX}"  # names are one-use-only in SageMaker
    est.fit({"train": TRAIN_CHANNEL}, job_name=job_name, wait=True)
    return f"s3://{bucket}/{prefix}/artifacts/training/{name}/{job_name}/output/model.tar.gz"


sklearn_runs = {
    "logreg-c1":  {"model": "logreg", "c-reg": "1.0"},
    "rf-d16-l5":  {"model": "rf", "n-estimators": "300", "max-depth": "16",
                   "min-samples-leaf": "5"},
}

### 5a. sklearn baselines (script mode, ml.m5.xlarge)

In [ ]:
model_artifacts = {}
for name, hp in sklearn_runs.items():
    print(f"=== {name} ===")
    model_artifacts[name] = run_training_job(name, SKLearn, dict(hp), "ml.m5.xlarge")

### 5b. XGBoost grid (ml.m5.2xlarge, primary model)

In [ ]:
for name, hp in XGB_GRID.items():
    print(f"=== {name} ===")
    model_artifacts[name] = run_training_job(name, Estimator, dict(hp),
                                             "ml.m5.2xlarge", image=XGB_IMAGE)
print(json.dumps(model_artifacts, indent=2))

## 6. 📊 Metrics comparison (screenshot for the tracker)

Pulls every `<model>_metrics.json` out of the model tarballs and ranks the
runs by validation macro F1 (champion rule: best validation macro F1 subject
to a High-Risk recall floor; notebook 04 confirms on the test split).

In [ ]:
import shutil


def fetch_metrics(artifact_s3):
    local_tar = Path("model.tar.gz")
    key = artifact_s3.split(f"s3://{bucket}/")[1]
    s3.download_file(bucket, key, str(local_tar))
    with tarfile.open(local_tar) as tar:
        member = next(m for m in tar.getnames() if m.endswith("_metrics.json"))
        metrics = json.loads(tar.extractfile(member).read())
    local_tar.unlink()
    return metrics


rows = []
for name, artifact_s3 in model_artifacts.items():
    m = fetch_metrics(artifact_s3)
    rows.append({
        "run": name,
        "model": m["model"],
        "val_macro_f1": m["validation"]["macro_f1"],
        "val_high_recall": m["validation"]["high_risk_recall"],
        "test_macro_f1": m["test"]["macro_f1"],
        "test_high_recall": m["test"]["high_risk_recall"],
        "test_roc_auc": m["test"]["roc_auc_ovr"],
    })
comparison = (pd.DataFrame(rows)
              .sort_values("val_macro_f1", ascending=False)
              .reset_index(drop=True))
comparison

# persist for notebook 04
json.dump(model_artifacts, open("artifacts/evaluation/model_artifacts.json", "w"), indent=2)
s3.upload_file("artifacts/evaluation/model_artifacts.json", bucket,
               f"{prefix}/artifacts/evaluation/model_artifacts.json")
comparison.to_csv("artifacts/evaluation/training_comparison.csv", index=False)
s3.upload_file("artifacts/evaluation/training_comparison.csv", bucket,
               f"{prefix}/artifacts/evaluation/training_comparison.csv")